# Week 7 - Bayesian Optimization

Generate optimized recommendations for Week 7 using utility modules.

## Setup

In [ ]:
import numpy as np
import warnings
import sys
import importlib
sys.path.append('..')  # Add parent directory to path

# Import utility modules (reload to pick up any code changes)
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import propose_next_point, fit_gp, get_strategy

from utils.data_utils import (
    load_week_data,
    save_week_data,
    combine_with_week_results, 
    print_data_summary
)

## 1. Load Week 6 Data

Load the combined data from previous weeks

In [ ]:
# Load Week 6 clean data
inputs, outputs = load_week_data("../week 6/week6_clean_data.npz")
print_data_summary(inputs, outputs, "Week 6 Data")

## 2. Add Week 6 Results

In [ ]:
# Week 6 submitted points
week6_inputs = {
    1: np.array([0.418000, 0.410000]),
    2: np.array([0.705000, 0.125362]),
    3: np.array([0.347863, 0.667420, 0.439172]),
    4: np.array([0.409529, 0.347548, 0.378665, 0.401944]),
    5: np.array([1.000000, 1.000000, 1.000000, 0.999000]),
    6: np.array([0.755469, 0.275580, 0.644099, 0.672228, 0.162862]),
    7: np.array([0.000000, 0.322263, 0.707844, 0.246481, 0.405689, 0.758028]),
    8: np.array([0.137475, 0.170302, 0.000000, 0.257071, 1.000000, 0.104132, 0.230000, 0.999373])
}

# Week 6 outputs (received from black box)
week6_outputs = {
    1: 0.7061894736746598,
    2: 0.5724979044516502,
    3: -0.00794683392719,
    4: 0.46567597434705066,
    5: 8643.147169534574,
    6: -0.5207421769045095,
    7: 1.85355113760358,
    8: 9.7401535257681
}

# Combine with Week 6 results
inputs, outputs = combine_with_week_results(inputs, outputs, week6_inputs, week6_outputs)
print_data_summary(inputs, outputs, "After Week 6 Results")

In [ ]:
# Save combined data for Week 8
save_week_data(inputs, outputs, "week7_clean_data.npz")

## 3. Week 6 Results Analysis

Evaluate which strategies worked and which failed to inform Week 7 approach.

In [ ]:
# Week 6 results analysis
print("=" * 70)
print("WEEK 6 RESULTS ANALYSIS")
print("=" * 70)

# Best values before Week 6 query
best_before_w6 = {}
for fid in range(1, 9):
    best_before_w6[fid] = np.max(outputs[fid][:-1])

print(f"\n{'F':>2} {'Dims':>4} {'Best Before W6':>14} {'W6 Query':>14} {'New Best':>14} {'Status'}")
print("-" * 70)

improved = 0
for fid in range(1, 9):
    dim = inputs[fid].shape[1]
    prev_best = best_before_w6[fid]
    w6_val = week6_outputs[fid]
    new_best = np.max(outputs[fid])
    
    if w6_val >= prev_best:
        status = "NEW BEST"
        improved += 1
    else:
        status = f"miss (best still {prev_best:.4f})"
    
    print(f"{fid:>2} {dim:>3}D {prev_best:>14.4f} {w6_val:>14.4f} {new_best:>14.4f}   {status}")

print(f"\nWeek 6 hit rate: {improved}/8 functions improved")
print("=" * 70)

## 4. Sensitivity Analysis

Updated sensitivity analysis with Week 6 data to inform Week 7 strategies.

In [ ]:
from utils.sensitivity import sensitivity_analysis

for func_id in range(1, 9):
    sensitivity_analysis(func_id, inputs[func_id], outputs[func_id])

## 5. Week 7 Strategy Design

Strategies based on 6-week performance history, updated sensitivity analysis, GP kernel diagnostics, peer results analysis, and failed strategy elimination.

### Week 6 lessons learned:
- **Manual nudges continue to dominate** — 3/8 new bests, all from manual strategies (F1, F6, F7)
- **F4 EI regressed hard** — scored 0.466 vs best 0.710, breaking 4-week improvement streak
- **F6 recovery confirmed** — locking dims 4&5 and nudging only dim2 recovered from W5's -0.99 disaster
- **F7 double nudge worked** — dim2-0.02 + dim6-0.02 gave 1.854 (new best)
- **F5 corner is confirmed** — [1,1,1,1] = 8662, any deviation loses points
- **F8 nearly converged** — UCB miss by only 0.02

### Peer intelligence:
- **F2 is bimodal** — a classmate found a second peak at **0.829** vs our best 0.614. We've been exploiting the wrong peak.
- **F3** — peer confirms −0.006 is competitive.
- **F5** — peers confirm boundary/corner strategy works.
- **F1** — peers stuck at ~10^-9. Our 0.706 is far ahead.

### Failed strategies to avoid:
- **F1:** Large moves kill (W2 dim2+0.27 → 3.1e-39)
- **F2:** Nothing beat W2's 0.614 in 4 weeks — W3 overshot dim1, W4 wrong region, W5–W6 micro-nudges all missed
- **F3:** Wide exploration failed (W3, W5). dim2-0.005 failed (W6)
- **F4:** EI can regress (W6 → 0.466)
- **F5:** Any deviation from [1,1,1,1] loses points (W5, W6)
- **F6:** Moving any dim except dim2 → disaster (W3, W4, W5). dim2+ micro-nudge nearly zero gain (W6)
- **F7:** dim2 too high (W5: 0.410) or too low (W4: 0.271) both fail
- **F8:** Multi-dim changes failed twice (W5, W6)

### Strategy per function:

| F | Strategy | Rationale | Avoids |
|---|----------|-----------|--------|
| F1 | **Manual** dim2+0.005 → [0.418, 0.415] | Alternate dims. W5's dim2+0.005 gave biggest jump (+47%). dim1 used in W6 — switch back. | Repeating W6's dim1 move |
| F2 | **EI wide** xi=0.5, full [0,1]^2 | Peer found 0.829 peak. Break out of failed exploitation loop. High xi forces exploration of new regions. | 4 weeks of failed strategies near dim1~0.7 |
| F3 | **Manual** dim1-0.001 → [0.347, 0.672, 0.439] | Follow improvement direction (dim1 decreased 0.493→0.348). Untested dim. Smaller step than W6's failed 0.005. | W6's dim2-0.005 failure, W3/W5 wide exploration |
| F4 | **Manual** dim3-0.005 → [0.414, 0.367, 0.360, 0.413] | dim3 decreased W4→W5 (0.405→0.365) with score jump. W6 EI pushed dim3 back up and regressed. | W6's EI failure |
| F5 | **EXPLORE** [0, 0, 1, 1] | [1,1,1,1] solved (8662, locked in). Resubmitting wastes query. Untested corner, zero downside. | Wasting query on known optimum, repeating W5/W6 micro-deviations |
| F6 | **Manual** dim2-0.005 → [0.755, 0.271, 0.644, 0.672, 0.163] | Test opposite direction. dim2+ gave near-zero gain in W6 (+0.0003). dim2=0.271 scored -0.521 in W2 — need to know if going below maps the landscape. | Repeating W6's dim2+ micro-nudge |
| F7 | **Manual** dim2-0.005 → [0.0, 0.317, 0.708, 0.246, 0.406, 0.758] | Isolate dim2 (W6 moved both dim2 and dim6). Smaller step. Stays in sweet spot (0.31-0.33). | W4's dim2=0.271 overshoot, W5's dim2=0.410 overshoot |
| F8 | **Manual** dim4+0.01 → [0.107, 0.120, 0.020, 0.217, 1.0, 0.100, 0.180, 0.987] | Only dim that rose consistently W2→W4 with improving scores. Single-dim only. | W5/W6 multi-dim failures |

In [ ]:
import warnings
import importlib
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import fit_gp, propose_next_point

week7_recommendations = {}

def get_best_point(fid):
    """Get the best observed point for a function"""
    best_idx = np.argmax(outputs[fid])
    return inputs[fid][best_idx].copy()

# ============================================================
# F1: Manual — dim2+0.005 (alternate from W6's dim1+0.005)
# W5's dim2+0.005 gave +47% (biggest jump). W6 did dim1. Alternate.
# Avoids: repeating W6's exact move. Large moves (W2 disaster).
# ============================================================
best1 = get_best_point(1)  # [0.418, 0.410] → 0.7062
week7_recommendations[1] = np.array([best1[0], best1[1] + 0.005])

# ============================================================
# F2: EI with wide bounds xi=0.5 — hunt second peak
# Peer found 0.829 peak. Our 0.614 is the weaker peak.
# 4 straight weeks of micro-nudges near [0.7, 0.125] all failed.
# High xi forces EI toward high-uncertainty unexplored regions.
# Avoids: repeating failed exploitation loop (W3-W6).
# ============================================================
bounds_f2 = np.array([[0.0, 1.0], [0.0, 1.0]])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pt2, _ = propose_next_point(
        inputs[2], outputs[2], bounds_f2,
        acq_func='EI', xi=0.5, n_restarts=50
    )
week7_recommendations[2] = pt2

# ============================================================
# F3: Manual — dim1-0.001 (follow improvement direction)
# Initial best dim1=0.493 → W4 best dim1=0.348. dim1 decreased.
# 0.001 = 1/4 of length scale (0.004). Conservative.
# Avoids: W6's dim2-0.005 failure, W3/W5 wide exploration disasters.
# ============================================================
best3 = get_best_point(3)  # [0.347863, 0.672420, 0.439172] → -0.0056
week7_recommendations[3] = np.array([best3[0] - 0.001, best3[1], best3[2]])

# ============================================================
# F4: Manual — dim3-0.005 (follow W4→W5 improvement trend)
# dim3: W4=0.405→W5=0.365, score: 0.672→0.710.
# W6 EI pushed dim3 back UP to 0.379 → regressed to 0.466.
# Continue downward: 0.365→0.360.
# Avoids: W6's EI failure. Trusting optimizer on this function.
# ============================================================
best4 = get_best_point(4)  # W5 best: [0.413690, 0.367443, 0.365391, 0.413441] → 0.7101
week7_recommendations[4] = np.array([best4[0], best4[1], best4[2] - 0.005, best4[3]])

# ============================================================
# F5: Explore [0, 0, 1, 1] — untested corner
# [1,1,1,1] = 8662 (locked in). Resubmitting = wasted query.
# W2: [0.38, 0.30, 1.0, 1.0] → 1688 shows low dim1&2 + high dim3&4 works.
# [0,0,1,1] extends that — maximum information, zero downside.
# Avoids: wasting query on known optimum, repeating W5/W6 micro-deviations.
# ============================================================
week7_recommendations[5] = np.array([0.000000, 0.000000, 1.000000, 1.000000])

# ============================================================
# F6: Manual — dim2-0.005 (test opposite direction)
# W6: dim2+0.005 gave near-zero gain (+0.0003). Stale strategy.
# W2 had dim2=0.271 → -0.521, W6 had dim2=0.276 → -0.521. Nearly same.
# Test dim2=0.271 to map the landscape below current best.
# Avoids: repeating W6's dim2+ micro-nudge. Moving dims 1/3/4/5 (disasters).
# ============================================================
best6 = get_best_point(6)  # [0.755469, 0.275580, 0.644099, 0.672228, 0.162862] → -0.5207
week7_recommendations[6] = np.array([
    best6[0],           # dim1 locked (ls=11.6)
    best6[1] - 0.005,   # dim2 -0.005 (test opposite direction)
    best6[2],           # dim3 locked (ls=24.5)
    best6[3],           # dim4 LOCKED (ls=0.001)
    best6[4]            # dim5 LOCKED (ls=0.0003)
])

# ============================================================
# F7: Manual — dim2-0.005 (isolate, safe step)
# W6: dim2=0.322 → 1.854 (best). Moved both dim2 AND dim6.
# Isolate dim2 to confirm it drove the improvement.
# dim2-0.005 → 0.317. Safe: well above W4's dim2=0.271 drop-off.
# Avoids: W4's overshoot (0.271), W5's overshoot (0.410), multi-dim confound.
# ============================================================
best7 = get_best_point(7)  # [0.0, 0.322263, 0.707844, 0.246481, 0.405689, 0.758028] → 1.854
week7_recommendations[7] = np.array([
    best7[0],            # dim1 locked at 0.0
    best7[1] - 0.005,    # dim2 -0.005 (0.322→0.317)
    best7[2],            # dim3 locked
    best7[3],            # dim4 locked
    best7[4],            # dim5 locked
    best7[5]             # dim6 locked (isolate dim2)
])

# ============================================================
# F8: Manual — dim4+0.01 (follow improving trend)
# dim4: W2=0.043→W3=0.127→W4=0.207 (only dim rising consistently).
# W5 dropped dim4 to 0.107 AND changed other dims → missed.
# W6 jumped dim4 to 0.257 AND changed other dims → missed.
# Single-dim +0.01 is conservative (0.003 length scales).
# Avoids: W5/W6 multi-dim failures.
# ============================================================
best8 = get_best_point(8)  # W4 best: [0.107, 0.120, 0.020, 0.207, 1.0, 0.100, 0.180, 0.987]
week7_recommendations[8] = np.array([
    best8[0],            # dim1 locked
    best8[1],            # dim2 locked
    best8[2],            # dim3 locked
    best8[3] + 0.010,    # dim4 +0.01 (follow rising trend)
    best8[4],            # dim5 locked at 1.0
    best8[5],            # dim6 locked (ls=1e5)
    best8[6],            # dim7 locked
    best8[7]             # dim8 locked (ls=7.9e4)
])

# ============================================================
# Sanity check all recommendations
# ============================================================
print("Week 7 Recommendations")
print("=" * 80)
for fid in range(1, 9):
    X, y = inputs[fid], outputs[fid]
    rec = week7_recommendations[fid]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gp = fit_gp(X, y)
    pred, pred_std = gp.predict(rec.reshape(1, -1), return_std=True)
    dists = np.linalg.norm(X - rec, axis=1)
    min_dist = np.min(dists)
    best = np.max(y)
    
    strategy = {
        1: "Manual dim2+0.005 (alternate from W6)",
        2: "EI wide xi=0.5 — hunt second peak",
        3: "Manual dim1-0.001 (follow trend, new dim)",
        4: "Manual dim3-0.005 (follow W4→W5 trend)",
        5: "EXPLORE [0,0,1,1] — untested corner",
        6: "Manual dim2-0.005 (test opposite direction)",
        7: "Manual dim2-0.005 (isolate, safe step)",
        8: "Manual dim4+0.01 (follow rising trend)"
    }
    
    print(f"F{fid} ({X.shape[1]}D)  best={best:.4f}  pred={pred[0]:.4f}±{pred_std[0]:.4f}  dist={min_dist:.4f}  {strategy[fid]}")
    print(f"  point: {rec}")
print("=" * 80)

## 6. Submission Format

In [ ]:
# Submission format
print("=" * 70)
print("WEEK 7 SUBMISSION")
print("=" * 70)

for fid in range(1, 9):
    pt = week7_recommendations[fid]
    formatted = '-'.join(f'{x:.6f}' for x in pt)
    print(f"Function {fid}:\t{formatted}")